In [205]:
import os
import numpy as np
import cv2 as cv
from skimage.io import imread
from skimage.filters import frangi
import matplotlib.pyplot as plt
from skimage.morphology import skeletonize, remove_small_objects
from scipy.ndimage import distance_transform_edt

from Skel2Graph3D import skel2graph3d
from Graph2Skel3D import graph2skel3d

# libraries to test
import sys
sys.path.append("../")
from frangi_2dfilter import frangi_2Dfilter
from fuzzy_thresholding import fuzzy_thresholding
from adaptive_thresholding import adaptive_thresholding

import pandas as pd

# Translated Code

In [206]:
import numpy as np

def pk_get_nh_idx(img, indices):
    """
    Python translation of MATLAB pk_get_nh_idx.
    - img: 3D numpy array (shape (W,H,D)). Should be padded externally (like MATLAB padarray).
    - indices: 1D array-like of linear indices (numpy 0-based indices into img.ravel()).
    Returns:
      nhood: (N,27) int64 array. Each row contains 27 0-based linear indices for that voxel's 3x3x3 neighborhood.
    """
    indices = np.asarray(indices, dtype=np.int64)
    if indices.ndim != 1:
        indices = indices.ravel()

    w, h, d = img.shape
    # unravel to coordinates (0-based)
    x, y, z = np.unravel_index(indices, (w, h, d))
    # prepare output
    n = len(indices)
    nhood = np.empty((n, 27), dtype=np.int64)

    col = 0
    # MATLAB loops xx=1:3,yy=1:3,zz=1:3 and uses x+xx-2 etc.
    # We loop offsets (-1,0,+1) which corresponds to xx-2 in MATLAB.
    for dx in (-1, 0, 1):
        for dy in (-1, 0, 1):
            for dz in (-1, 0, 1):
                # neighbor coordinates
                nx = x + dx
                ny = y + dy
                nz = z + dz
                # convert to linear index (0-based)
                nhood[:, col] = np.ravel_multi_index((nx, ny, nz), (w, h, d))
                col += 1

    return nhood


### Observations:
This code returns a 0-indexed matrix!!
If we don't want this, we can always run thru the matrix and update each value +1, or during runtime if possible 

### Testing
```
img = np.zeros((4, 4, 4), dtype=np.uint32)
(set some values in img)
i = np.flatnonzero(img)
```

### Usage
We will be dneeding nhi for building s2n later (s2n is the linear array where the ith = flat_idx of voxel, and where cans[s2n[i]] = cannal neighbors of voxel)

In [207]:
def pk_get_nh(img, indices):
    """
    Python translation of MATLAB pk_get_nh.
    - img: 3D numpy array (dtype can be bool or int). Should be padded externally.
    - indices: 1D array-like of linear indices (0-based).
    Returns:
      nhood: (N,27) boolean numpy array; True if voxel is foreground/non-zero.
    """
    indices = np.asarray(indices, dtype=np.int64).ravel()
    w, h, d = img.shape
    x, y, z = np.unravel_index(indices, (w, h, d))

    n = len(indices)
    nhood = np.empty((n, 27), dtype=bool)

    col = 0
    for dx in (-1, 0, 1):
        for dy in (-1, 0, 1):
            for dz in (-1, 0, 1):
                nx = x + dx
                ny = y + dy
                nz = z + dz
                linear_idx = np.ravel_multi_index((nx, ny, nz), (w, h, d))
                # pick values from img by flat indexing
                nhood[:, col] = img.ravel()[linear_idx].astype(bool)
                col += 1

    return nhood

## Observations:
There are shifts in the positionings due to the column-major -> row-major shift
The resulting matrices (output) are not the same in matlab than in python; however, since this matrix is used for lookups... As long as in python we use the      np.flatten method to access linearly each voxel, we should be fine. If we also myltiply each row of nh_idx and nh we should be left with a propper lookup matrix for each voxel's neighbors with respect to the skeletonized image.

### Testing
```
img = np.zeros((4, 4, 4), dtype=np.uint32)
(set some values in img)
i = np.flatnonzero(img)

img → 3D binary array (logical), the skeletonized image.

p1 → starting voxel linear index.

p2 → target voxel index or sentinel.

canal → logical mask of voxels that are neither node nor endpoint (the “canal”).

nh → 26×N matrix of neighborhood indices for each voxel (like from pk_get_nh_idx).

idx → flattened indices of all voxels in img.

Used to cateogrize types of voxels
Used to find all potential unvisited links emanating from a given start_node, when path finding with pk_follow_link

function [vox,n_idx,ep] = pk_follow_link(skel,node,k,j,idx,cans,c2n)

vox = [];
n_idx = [];
ep = 0;

% assign start node to first voxel
vox(1) = node(k).idx(j);

i=1;
isdone = false;
while(~isdone) % while no node reached
    i=i+1; % next voxel
    next_cand = c2n(idx);
        cand = cans(next_cand,2);
        if(cand==vox(i-1)) % switch direction
            cand = cans(next_cand,3);
        end;
        if(skel(cand)>1) % node found
            vox(i) = idx;
            vox(i+1) = cand; % first node
            n_idx = skel(cand)-1; % node #
            if(node(n_idx).ep)
                ep=1;
            end;
            isdone = 1;
        else % next voxel
            vox(i) = idx;
            idx = cand;
        end;
end;


In [ ]:
def pk_follow_link(skel2, nodes, curr_node_cluster_idx, curr_node_idx, can_idx, cans, c2n):
    '''
    Returns the the path,  from the current node and its current cannal voxel neighbor to
    another node (endpoint or not)

    Inupts: 
        - skel2 [np.ndarray]: skeletonized image with identified node clusters (clusters have unique id [2, inf], all 1s are
            cannal voxels, and all 0s are background voxels)
        - nodes [list[node_cluster]]: list of all node_clusters in the network
        - 
    '''
    path = []
    ep = 0

    path.append(nodes[curr_node_cluster_idx].nodes_idx[curr_node_idx])

    done = False
    while not done:
        next_cand = c2n[can_idx] - 1
        cand = cans[next_cand][1]

        # swtich to other cannal voxel neighbor
        if cand == path[-1]:
            cand = cans[next_cand][2]

        if skel2.flat[cand] > 1:
            path.append(can_idx)
            path.append(int(cand))
            n_idx = skel2.flat[cand] - 2

            ep = nodes[n_idx].ep
            done = True

        else:
            path.append(curr_node_idx)
            curr_node_idx = cand

    return (path, n_idx, ep)

# path = pk_follow_link(skel2, nodes, 0, 0, 22, cans, c2n)

# An example for testing pk_functions

In [211]:
# Create a 4x4 uint32 array of zeros
test_skel = np.zeros((4, 4), dtype=np.uint32)
skel2 = np.zeros((4, 4), dtype=np.uint32)

# Set specific pixels to 1
test_skel[0, 0] = 1      # (1,1) in MATLAB
test_skel[1, 0] = 1      # (2,1)
test_skel[1, 1] = 1      # (2,2)
test_skel[2, 0] = 1      # (3,1)
test_skel[1, 2] = 1      # (2,3)
test_skel[3, 3] = 1      # (4,4)

skel2[0, 0] = 1
skel2[1, 0] = 2
skel2[2, 0] = 1
skel2[1, 1] = 2
skel2[1, 2] = 1
skel2[3, 3] = 1


# Add a 3rd dimension (equivalent to test_skel(:, :, 1) in MATLAB)
test_skel = test_skel[:, :, np.newaxis]
skel2 = skel2[:, :, np.newaxis]

skel = np.pad(test_skel, pad_width=1, mode='constant', constant_values=0)
skel2 = np.pad(skel2, pad_width=1, mode='constant', constant_values=0)

list_canal = np.flatnonzero(skel)

nh = pk_get_nh(skel, list_canal)
nhi = pk_get_nh_idx(skel, list_canal)

sum_nh = np.sum(nh.astype(bool), axis=1)

nodes = list_canal[sum_nh > 3]
ep = list_canal[sum_nh == 2]
cans = list_canal[sum_nh == 3]

can_nh_idx = pk_get_nh_idx(skel, cans)
can_nh = pk_get_nh(skel, cans)

# remove center of 3x3x3 cube (position 14 in MATLAB → index 13 in Python)
can_nh_idx = np.delete(can_nh_idx, 13, axis=1)
can_nh = np.delete(can_nh, 13, axis=1)

can_nb = np.sort(can_nh.astype(bool) * can_nh_idx, axis=1)

# remove zeros (keep only the last 2 columns)
can_nb = can_nb[:, -2:]

cans = np.column_stack((cans, can_nb))

# c2n: map each canal voxel index (from cans[:, 0]) to its row number in 'cans'
c2n = np.zeros(6 * 6 * 3, dtype=int)
c2n[cans[:, 0]] = np.arange(1, cans.shape[0] + 1)

# s2n: map each neighborhood center voxel (from nhi[:, 13]) to its row number in 'nhi'
s2n = np.zeros(6 * 6 * 3, dtype=int)
s2n[nhi[:, 13]] = np.arange(1, nhi.shape[0] + 1)

link_idx = s2n[[40, 43]] - 1

link_cands = nhi[link_idx[1], nh[link_idx[1], :] == 1]

ep_cands = np.intersect1d(link_cands, ep)


# filter for cannal voxels connected to curr node (link candidates / possible paths)
coords = np.unravel_index(link_cands, skel2.shape)
mask = skel2[coords] == 1
link_cands = link_cands[mask]

link_cands = np.intersect1d(link_cands, cans[:, 0])

In [212]:
class node_cluster:
    def __init__(self, nodes_idx: list[int], ep: int):
        self.nodes_idx = nodes_idx
        self.ep = ep

In [213]:
node = node_cluster([40, 43], ep = 0)
nodes = [node]

In [214]:
path = pk_follow_link(skel2, nodes, 0, 0, 22, cans, c2n)

In [220]:
path

([40, 22, 43], np.uint32(0), 0)

# Log: 10/16/25

### Possible bug/attention
When using s2n
Be careful because s2n[idx] will give the index-1 with which to access the neighbors of idx. So, we need to add some displacement of some sort

## Notes
- The code above is ready for pk_follow_link
- The translation is not fully performed, there are missing pieces of code that must be handleled later

Note: s2n[idx] - 1 is what really we use to access the correct negibhors (nhi) of the idx node 

In [216]:
import numpy as np

def get_neighbor_offsets():
    """Return 26 neighbor offsets for 3D 6-connected or 26-connected neighborhood."""
    offsets = []
    for dz in [-1, 0, 1]:
        for dy in [-1, 0, 1]:
            for dx in [-1, 0, 1]:
                if dx == dy == dz == 0:
                    continue
                offsets.append((dz, dy, dx))
    return np.array(offsets)

def pk_follow_link(img, start_idx, canal_mask, neighbor_offsets):
    """
    Follow a canal (skeleton path) starting from a given voxel until
    reaching a node or endpoint.

    Parameters
    ----------
    img : np.ndarray (3D bool)
        Skeletonized binary image.
    start_idx : int
        Linear index (flattened) of the starting voxel.
    canal_mask : np.ndarray (3D bool)
        Voxels that are part of a canal (not nodes or endpoints).
    neighbor_offsets : np.ndarray of shape (26, 3)
        Relative offsets to find 26-neighbors.

    Returns
    -------
    path : list of int
        Sequence of linear voxel indices traversed along the canal.
    """
    visited = set()
    path = []
    dims = img.shape
    zyx = np.unravel_index(start_idx, dims)
    current = np.array(zyx)

    while True:
        path.append(np.ravel_multi_index(current, dims))
        visited.add(tuple(current))

        # get neighbors
        neighbors = current + neighbor_offsets
        valid = (
            (neighbors[:, 0] >= 0) & (neighbors[:, 0] < dims[0]) &
            (neighbors[:, 1] >= 0) & (neighbors[:, 1] < dims[1]) &
            (neighbors[:, 2] >= 0) & (neighbors[:, 2] < dims[2])
        )
        neighbors = neighbors[valid]

        # filter neighbors: canal and unvisited
        neighbor_indices = np.ravel_multi_index(neighbors.T, dims)
        valid_neighbors = [
            n for n in neighbor_indices
            if canal_mask.flat[n] and tuple(np.unravel_index(n, dims)) not in visited
        ]

        if not valid_neighbors:
            break  # reached end or node

        # move to next neighbor (should only be one in a clean skeleton)
        current = np.array(np.unravel_index(valid_neighbors[0], dims))

    return path


In [217]:
skel2 = skel.copy()
skel2[2][1][1] = 2
skel2[2][2][1] = 2
skel2

array([[[0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0]],

       [[0, 0, 0],
        [0, 1, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0]],

       [[0, 0, 0],
        [0, 2, 0],
        [0, 2, 0],
        [0, 1, 0],
        [0, 0, 0],
        [0, 0, 0]],

       [[0, 0, 0],
        [0, 1, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0]],

       [[0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 1, 0],
        [0, 0, 0]],

       [[0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0]]], dtype=uint32)

In [218]:
path = pk_follow_link(skel2, 22, cans, link_cands[0], cans, c2n)
path

TypeError: pk_follow_link() takes 4 positional arguments but 6 were given

# Finalized Plan
- Ensure that pk_get_nh and pk_get_nh_idx return matrices that have the exact same values as the matlab function
    - Fix the col-major -> row-major mismatch
    - Fix the 0-indexing issue (enforce a 1-indexing to keep it as consistent as possible) (try t)
- Ensure that skel2 array correctly labels nodes and cannal voxels
- Ensure that pk_follow_link updates the skel2 array by setting visited cannal voxels to 0

In [ ]:
from typing import Any
import numpy as np
from scipy import sparse
from skimage.measure import label
from scipy.ndimage import label as scp_label
from pk_functions import pk_get_nh, pk_get_nh_idx, pk_follow_link


# label connected components 
lm = label(skel, connectivity=3)

# get image dimensions
w, l, h = skel.shape
skel2 = skel.astype(np.uint16) # needed for labeling nodes etc.

# all foreground (True) voxels, as flat indices
list_canal = np.flatnonzero(skel)

# if image contains no skeletonized data, return empty elements
if len(list_canal) == 0:
    raise ValueError("No skeletonized data: the input 3D mask contains no foreground voxels.")

# 26-nh of all canal voxels
nh = pk_get_nh(skel, list_canal)       # shape: (N, 27)

# 26-nh indices of all canal voxels
nhi = pk_get_nh_idx(skel, list_canal)  # shape: (N, 27)

# num of 26-nb of each skel voxel + 1
sum_nh = np.sum(nh.astype(bool), axis=1)

# node/endpoint/canal assignment
nodes = list_canal[sum_nh > 3] # all canal voxels with >2 nb are nodes -> branching points
ep    = list_canal[sum_nh == 2] # all canal voxels with exactly one nb are end nodes -> endpoints
cans  = list_canal[sum_nh == 3] # all canal voxels with exactly 2 nb -> segments

# Nx3 matrix with the 2 neighbors of each canal voxel
if cans.size > 0:
    can_nh_idx = pk_get_nh_idx(skel, cans)
    can_nh = pk_get_nh(skel, cans)

    # remove center of 3x3 cube
    can_nh_idx = np.delete(can_nh_idx, 13, axis=1)
    can_nh = np.delete(can_nh, 13, axis=1)

    # keep only the two existing foreground voxels
    can_nb = np.sort(can_nh * can_nh_idx, axis=1)
    can_nb = can_nb[:, -2:]  # keep last two columns

    # add neighbours to canalicular voxel list (this might include nodes)
    cans_all = np.column_stack([cans, can_nb])
else:
    cans_all = np.empty((0, 3), dtype=int)

# group clusters of node voxels to nodes
node, link = [], []

tmp = np.zeros((w, l, h), dtype=bool)
tmp.ravel()[nodes] = True
labelimg, num_realnodes = scp_label(tmp, structure=np.ones((3,3,3))) # number of unique nodes

# create node structure for branching points (node metadata)
for labelidx in range(1, num_realnodes+1):
    idxs = np.flatnonzero(labelimg == labelidx)
    x, y, z = np.unravel_index(idxs, (w,l,h))
    node.append({
        'idx': idxs.tolist(),
        'links': [], # indices to access each link's metadata (defined below)
        'conn': [],
        'comx': float(np.mean(x)),
        'comy': float(np.mean(y)),
        'comz': float(np.mean(z)),
        'ep': 0,
        'label': int(lm.flat[idxs[0]]),
    })
    skel2.flat[idxs] = len(node)  # assign index to node voxels

# cluster endpoint voxels ('ep')
tmp = np.zeros((w, l, h), dtype=bool)
tmp.ravel()[ep] = True
labelimg, num_endnodes = scp_label(tmp, structure=np.ones((3,3,3)))
for idx in range(1, num_endnodes+1):
    idxs = np.flatnonzero(labelimg == idx)
    ni = num_realnodes + idx - 1
    x, y, z = np.unravel_index(idxs, (w, l, h))

    # create node structure for endpoints (node metadata)
    node.append({
        'idx': idxs.tolist(),
        'links': [], # indices to access each link's metadata (defined below)
        'conn': [],
        'comx': float(np.mean(x)),
        'comy': float(np.mean(y)),
        'comz': float(np.mean(z)),
        'ep': 1,
        'label': int(lm.flat[idxs[0]]),
    })
    skel2.flat[idxs] = len(node)
    
l_idx = 0

c2n = np.zeros(w*l*h, dtype=int)
if cans_all.shape[0] > 0:
    c2n[cans_all[:,0]] = np.arange(cans_all.shape[0])
s2n = np.zeros(w*l*h, dtype=int)
if nhi.shape[0] > 0:
    s2n[nhi[:,13]] = np.arange(nhi.shape[0])

In [ ]:
skel2

array([[[0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0]],

       [[0, 0, 0],
        [0, 1, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0]],

       [[0, 0, 0],
        [0, 1, 0],
        [0, 1, 0],
        [0, 2, 0],
        [0, 0, 0],
        [0, 0, 0]],

       [[0, 0, 0],
        [0, 1, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0]],

       [[0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 1, 0],
        [0, 0, 0]],

       [[0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0]]], dtype=uint16)

In [ ]:
cans_all

array([[22, 40, 43],
       [58, 40, 43]])

In [ ]:
class Node:
    def __init__(self, value = None, next = None):
        self.val = value
        self.next = next

head = Node(1, Node(2, Node(3)))

In [ ]:
head.next.val

2